# USA — Already-on-BFP APT Validation

## Purpose

Validate a set of **scoped APT ids** (USA) against the current layer `14533` and today's
Building Footprint (BFP), then emit the Data Correctness (DC) input format for the ones that
qualify.

## Steps

1. **Load snapshot for layer `14533`** and filter to USA address points; display the row count. Dataset name: `latestAptDS`.
2. **Load BFP** from catalog `preprocess_prod.bfp`. Dataset name: `usaBfpDataset`.
3. **Load the scoped CSV file(s)** as `scopedDS`.
4. **Join `scopedDS` with `latestAptDS`** on the APT ids → `joinedDS`.
5. **Enrich `joinedDS`:** add `latest_vs_scoped_distance_in_meters` (distance between the `latestAptDS` point and the `scopedDS` point) and `location_change` (true/false, whether the geometry differs between latest and scoped).
6. **Join `joinedDS` with `usaBfpDataset`:** add `is_intersects_with_bfp` (true if the APT intersects any BFP) and `bfp_geom` (the intersecting BFP polygon).
7. **Filter** to rows that intersect a BFP.
8. **Format** the scoped output in the Data Correctness (DC) job input format.

In [ ]:
%sql
DROP DATABASE IF EXISTS usa_alredy_on_bfp_validation CASCADE;
CREATE DATABASE IF NOT EXISTS usa_alredy_on_bfp_validation;

In [ ]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "USA"
val NEW_DATABASE = "usa_alredy_on_bfp_validation"
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_new"

In [ ]:
%scala
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = USA
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")

In [ ]:
%scala
// Read the USA Building Footprint (BFP) polygons from preprocess_prod.bfp.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cell
val sedona = SedonaContext.create(spark)

val usaBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === COUNTRY_ISO)
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

println(s"USA BFP polygon count: ${usaBfpDataset.count()}")
display(usaBfpDataset)